# Eye Disease — Balanced Ensemble (T4 Optimised)
### EfficientNetV2-M + CBAM  ·  ConvNeXt-Base  ·  ViT-Base/16
**Target: ~5-7 min/epoch · ~4-5 hrs total**

| Model | Params | Role |
|---|---|---|
| EfficientNetV2-M + CBAM | ~54M | Efficient CNN + spatial attention |
| ConvNeXt-Base | ~89M | Modern CNN, different feature patterns |
| ViT-Base/16 | ~86M | Global attention, catches spread-out pathology |


In [ ]:
# 1. Install and Imports
!pip install kagglehub timm -q

import os, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy

from PIL import Image
import torch, timm
import torch.nn as nn
import torch.nn.functional as F
from torch import tensor, long
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    efficientnet_v2_m, EfficientNet_V2_M_Weights,
    convnext_base,     ConvNeXt_Base_Weights,
)
from torchvision.transforms import (
    Compose, ToTensor, Normalize, Resize, CenterCrop,
    RandomHorizontalFlip, RandomRotation, RandomAffine,
    ColorJitter, GaussianBlur, RandomApply,
)
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark           = True
    torch.backends.cuda.matmul.allow_tf32    = True
    torch.backends.cudnn.allow_tf32          = True
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('CPU mode')
print('Imports ready.')


GPU  : Tesla T4
VRAM : 14.6 GB
Imports ready.


In [ ]:
# 2. Download Dataset
import kagglehub

path = kagglehub.dataset_download('shufanshahi/eye-disease-image-dataset-for-finetunning')
print('Dataset path:', path)

base = Path(path) / 'split_dataset'
train_dir, val_dir, test_dir = base/'train', base/'val', base/'test'

for split, d in [('TRAIN', train_dir), ('VAL', val_dir), ('TEST', test_dir)]:
    classes = sorted(os.listdir(d)) if d.exists() else []
    total   = sum(len(list((d/c).iterdir())) for c in classes if (d/c).is_dir())
    print(f'  {split:5s}: {len(classes)} classes, {total:,} images')


100%|██████████| 5.29G/5.29G [01:21<00:00, 69.6MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/shufanshahi/eye-disease-image-dataset-for-finetunning/versions/3
  TRAIN: 10 classes, 14,933 images
  VAL  : 10 classes, 533 images
  TEST : 10 classes, 1,068 images


In [ ]:
# 3. Class Mapping and DataFrames
CLASS_MAPPING = {
    'Central Serous Chorioretinopathy': 0,
    'Diabetic Retinopathy':             1,
    'Disc Edema':                       2,
    'Glaucoma':                         3,
    'Healthy':                          4,
    'Macular Scar':                     5,
    'Myopia':                           6,
    'Pterygium':                        7,
    'Retinal Detachment':               8,
    'Retinitis Pigmentosa':             9,
}
CLASS_NAMES = list(CLASS_MAPPING.keys())
NUM_CLASSES  = len(CLASS_NAMES)

def build_df(directory):
    rows = []
    for cls_name, label in CLASS_MAPPING.items():
        cls_dir = directory / cls_name
        if not cls_dir.exists(): continue
        for f in cls_dir.iterdir():
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                rows.append({'image': str(f), 'label': label})
    return pd.DataFrame(rows)

train_df = build_df(train_dir)
val_df   = build_df(val_dir)
test_df  = build_df(test_dir)
print(f'Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}')


Train 14,933 | Val 533 | Test 1,068


In [ ]:
# 4. Dataset and DataLoaders
# 288px: sweet spot for Medium models on T4
# larger than 224 (better features) but fits batch=24 in 16GB VRAM
IMG_SIZE = 288
BATCH    = 24
NWORK    = 4

_norm = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tf = Compose([
    Resize((IMG_SIZE + 24, IMG_SIZE + 24)),
    RandomRotation(15),
    RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    CenterCrop(IMG_SIZE),
    RandomHorizontalFlip(),
    ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    RandomApply([GaussianBlur(3)], p=0.2),
    ToTensor(), _norm,
])
val_tf = Compose([Resize((IMG_SIZE, IMG_SIZE)), ToTensor(), _norm])


class EyeDataset(Dataset):
    def __init__(self, df, training=True):
        self.df = df.reset_index(drop=True)
        self.tf = train_tf if training else val_tf
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:   img = Image.open(row['image']).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        return {'image': self.tf(img), 'label': tensor(row['label'], dtype=long)}


_kw = dict(num_workers=NWORK, pin_memory=USE_AMP,
           prefetch_factor=2, persistent_workers=True)
train_loader = DataLoader(EyeDataset(train_df, True),  BATCH, shuffle=True,  drop_last=True,  **_kw)
val_loader   = DataLoader(EyeDataset(val_df,   False), BATCH, shuffle=False, **_kw)
test_loader  = DataLoader(EyeDataset(test_df,  False), BATCH, shuffle=False, **_kw)

print(f'Batch={BATCH} | IMG={IMG_SIZE}px')
print(f'Batches: train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}')
t0 = time.time()
_ = next(iter(train_loader))
print(f'First batch: {time.time()-t0:.2f}s  (should be <2s)')


Batch=24 | IMG=288px
Batches: train 622 | val 23 | test 45
First batch: 4.86s  (should be <2s)


In [ ]:
# 5. CBAM Attention Module
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(ch, ch//r, 1, bias=False), nn.ReLU(),
            nn.Conv2d(ch//r, ch, 1, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv(torch.cat([avg, mx], 1)))

class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(ch, r)
        self.sa = SpatialAttention(k)
    def forward(self, x):
        return x * self.sa(x * self.ca(x))

print('CBAM ready.')


CBAM ready.


In [ ]:
# 6. Model Definitions

class EfficientModel(nn.Module):
    '''EfficientNetV2-M + CBAM attention on final feature map (~54M params).'''
    def __init__(self, n=NUM_CLASSES):
        super().__init__()
        b = efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
        self.features = b.features
        ch = 1280
        self.cbam = CBAM(ch)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(ch, 768),
            nn.BatchNorm1d(768), nn.ReLU(),
            nn.Dropout(0.3),    nn.Linear(768, n),
        )
    def forward(self, x):
        return self.head(self.pool(self.cbam(self.features(x))).flatten(1))


class ConvNeXtModel(nn.Module):
    '''ConvNeXt-Base: modern CNN with depthwise conv + LayerNorm (~89M params).'''
    def __init__(self, n=NUM_CLASSES):
        super().__init__()
        b = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.features = b.features
        self.avgpool  = b.avgpool
        ch = 1024
        self.head = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(ch),
            nn.Dropout(0.4), nn.Linear(ch, 768),
            nn.GELU(),
            nn.Dropout(0.3), nn.Linear(768, n),
        )
    def forward(self, x):
        return self.head(self.avgpool(self.features(x)))


class ViTModel(nn.Module):
    '''ViT-Base/16: global self-attention via timm (~86M params).'''
    def __init__(self, n=NUM_CLASSES):
        super().__init__()
        self.vit = timm.create_model(
            'vit_base_patch16_224', pretrained=True,
            num_classes=0, img_size=IMG_SIZE)
        ch = self.vit.embed_dim
        self.head = nn.Sequential(
            nn.LayerNorm(ch),
            nn.Dropout(0.4), nn.Linear(ch, 512),
            nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, n),
        )
    def forward(self, x):
        return self.head(self.vit(x))


print('Model   Params   Est. VRAM @ batch 24')
print('-' * 48)
for name, cls, vram in [
    ('EfficientNetV2-M+CBAM', EfficientModel, '~6 GB'),
    ('ConvNeXt-Base',         ConvNeXtModel,  '~8 GB'),
    ('ViT-Base/16',           ViTModel,       '~7 GB'),
]:
    m = cls()
    p = sum(x.numel() for x in m.parameters()) / 1e6
    print(f'  {name:<26} {p:6.1f}M   {vram}')
    del m
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('T4 VRAM: 16 GB - all models fit comfortably')


Model   Params   Est. VRAM @ batch 24
------------------------------------------------
Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth


100%|██████████| 208M/208M [00:01<00:00, 163MB/s]


  EfficientNetV2-M+CBAM        54.1M   ~6 GB
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:03<00:00, 115MB/s]


  ConvNeXt-Base                88.4M   ~8 GB


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  ViT-Base/16                  86.3M   ~7 GB
T4 VRAM: 16 GB - all models fit comfortably


In [ ]:
# 7. Hyperparameters
EPOCHS        = 20
LR            = 2e-4
WEIGHT_DECAY  = 3e-5
GRAD_CLIP     = 1.0
PATIENCE      = 6
FREEZE_EPOCHS = 3   # head-only warmup before full fine-tune

print(f'Epochs={EPOCHS} | LR={LR} | Batch={BATCH} | IMG={IMG_SIZE}px')
print(f'Freeze warmup={FREEZE_EPOCHS} epochs | Early stop patience={PATIENCE}')
print('Estimated total time on T4: 3-4 hrs (with early stopping)')


Epochs=20 | LR=0.0002 | Batch=24 | IMG=288px
Freeze warmup=3 epochs | Early stop patience=6
Estimated total time on T4: 3-4 hrs (with early stopping)


In [ ]:
# 8. Training Function with Freeze Warmup + Differential LRs
def set_backbone_grad(model, requires_grad):
    for name, p in model.named_parameters():
        if 'head' not in name:
            p.requires_grad = requires_grad


def make_opt_sch(model, epochs_remaining):
    backbone_p, cbam_p, head_p = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if   'cbam' in name: cbam_p.append(p)
        elif 'head' in name: head_p.append(p)
        else:                backbone_p.append(p)

    groups = [{'params': head_p, 'lr': LR}]
    if cbam_p:     groups.append({'params': cbam_p,     'lr': LR * 1.5})
    if backbone_p: groups.append({'params': backbone_p, 'lr': LR * 0.1})

    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g['lr'] for g in groups],
        steps_per_epoch=len(train_loader),
        epochs=epochs_remaining, pct_start=0.1, anneal_strategy='cos')
    return opt, sch


def train_one_model(model, model_name):
    model = model.to(device)

    if hasattr(torch, 'compile'):
        try:
            model = torch.compile(model, mode='reduce-overhead')
            print(f'  torch.compile applied to {model_name}')
        except Exception: pass

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler() if USE_AMP else None
    history   = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc, best_state, wait = 0.0, None, 0
    opt, sch = None, None

    for epoch in range(EPOCHS):
        if epoch == 0:
            set_backbone_grad(model, False)
            opt, sch = make_opt_sch(model, EPOCHS)
            print(f'  Backbone FROZEN - head warmup for {FREEZE_EPOCHS} epochs')
        elif epoch == FREEZE_EPOCHS:
            set_backbone_grad(model, True)
            opt, sch = make_opt_sch(model, EPOCHS - FREEZE_EPOCHS)
            print(f'  Backbone UNFROZEN - full fine-tuning with differential LRs')

        # Train
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        t_start = time.time()

        pbar = tqdm(train_loader,
                    desc=f'[{model_name}] E{epoch+1:02d}/{EPOCHS} Train', leave=False)
        for batch in pbar:
            imgs   = batch['image'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            if USE_AMP:
                with autocast():
                    out  = model(imgs)
                    loss = criterion(out, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()

            sch.step()
            t_loss    += loss.item()
            t_total   += labels.size(0)
            t_correct += (out.detach().argmax(1) == labels).sum().item()
            pbar.set_postfix(loss=f'{loss.item():.3f}')

        epoch_mins = (time.time() - t_start) / 60
        train_acc  = 100 * t_correct / t_total
        train_loss = t_loss / len(train_loader)

        # Validate
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader,
                              desc=f'[{model_name}] E{epoch+1:02d}/{EPOCHS} Val',
                              leave=False):
                imgs   = batch['image'].to(device, non_blocking=True)
                labels = batch['label'].to(device, non_blocking=True)
                if USE_AMP:
                    with autocast(): out = model(imgs); loss = criterion(out, labels)
                else:
                    out = model(imgs); loss = criterion(out, labels)
                v_loss    += loss.item()
                v_total   += labels.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()

        val_acc  = 100 * v_correct / v_total
        val_loss = v_loss / len(val_loader)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'  E{epoch+1:02d} | train {train_acc:.1f}% loss {train_loss:.4f} | '
              f'val {val_acc:.1f}% loss {val_loss:.4f} | {epoch_mins:.1f} min/epoch')

        if val_acc > best_val_acc + 0.05:
            best_val_acc = val_acc
            best_state   = deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            wait = 0
            print(f'    New best: {best_val_acc:.2f}%')
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'    Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    model.cpu()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f'[{model_name}] Done - best val: {best_val_acc:.2f}%')
    return model, history

print('Training function ready.')


Training function ready.


In [ ]:
# 9. Train All Three Models Sequentially
MODELS_TO_TRAIN = [
    ('efficientnet', EfficientModel, 'EfficientNetV2-M+CBAM'),
    ('convnext',     ConvNeXtModel,  'ConvNeXt-Base'),
    ('vit',          ViTModel,       'ViT-Base/16'),
]

trained, histories = {}, {}
total_start = time.time()

for key, model_cls, label in MODELS_TO_TRAIN:
    print('=' * 55)
    print(f'  Training: {label}')
    print('=' * 55)
    t0 = time.time()
    m, h = train_one_model(model_cls(), label)
    elapsed = (time.time() - t0) / 60
    trained[key]   = m
    histories[key] = h
    torch.save(m.state_dict(), f'{key}.pth')
    print(f'  Saved {key}.pth  ({elapsed:.1f} min)\n')

total_hrs = (time.time() - total_start) / 3600
print(f'All models trained in {total_hrs:.2f} hrs.')


  Training: EfficientNetV2-M+CBAM
  torch.compile applied to EfficientNetV2-M+CBAM
  Backbone FROZEN - head warmup for 3 epochs


[EfficientNetV2-M+CBAM] E01/20 Train:   0%|          | 0/622 [00:00<?, ?it/s]W0318 22:17:11.080000 18380 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  E01 | train 35.9% loss 1.9112 | val 52.3% loss 1.5932 | 12.7 min/epoch
    New best: 52.35%


  E02 | train 53.1% loss 1.5358 | val 52.5% loss 1.5307 | 10.4 min/epoch
    New best: 52.53%


  E03 | train 54.7% loss 1.4916 | val 55.7% loss 1.4587 | 10.3 min/epoch
    New best: 55.72%
  Backbone UNFROZEN - full fine-tuning with differential LRs


...

  E01 | train 42.2% loss 1.7700 | val 54.2% loss 1.4754 | 10.9 min/epoch
    New best: 54.22%


  E02 | train 57.1% loss 1.4365 | val 61.7% loss 1.2983 | 10.1 min/epoch
    New best: 61.73%


  E03 | train 60.7% loss 1.3641 | val 65.5% loss 1.2260 | 10.1 min/epoch
    New best: 65.48%
  Backbone UNFROZEN - full fine-tuning with differential LRs


  E04 | train 61.7% loss 1.3299 | val 67.0% loss 1.2007 | 10.0 min/epoch
    New best: 66.98%


[ConvNeXt-Base] E05/20 Train:  28%|██▊       | 174/622 [02:55<07:08,  1.05it/s, loss=1.121]

In [ ]:
# 10. Collect Softmax Probabilities
LABELS = {
    'efficientnet': 'EfficientNetV2-M+CBAM',
    'convnext':     'ConvNeXt-Base',
    'vit':          'ViT-Base/16',
}

def get_probs(model, loader, label=''):
    model = model.to(device).eval()
    probs_list, tgt_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Probs {label}', leave=False):
            imgs = batch['image'].to(device, non_blocking=True)
            if USE_AMP:
                with autocast(): logits = model(imgs)
            else:
                logits = model(imgs)
            probs_list.append(F.softmax(logits, dim=1).cpu())
            tgt_list.append(batch['label'])
    model.cpu()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return torch.cat(probs_list).numpy(), torch.cat(tgt_list).numpy()


all_probs = {}
targets   = None
for key, label in LABELS.items():
    probs, tgts = get_probs(trained[key], test_loader, label)
    all_probs[key] = probs
    if targets is None: targets = tgts

print('Probabilities collected.')


NameError: name 'trained' is not defined

In [ ]:
# 11. Individual Model Results
ind_results = {}
print(f'  Model                            Accuracy    Macro F1')
print('-' * 58)
for key, label in LABELS.items():
    preds = all_probs[key].argmax(axis=1)
    acc   = accuracy_score(targets, preds)
    _, _, f1, _ = precision_recall_fscore_support(
        targets, preds, average='macro', zero_division=0)
    ind_results[key] = {
        'accuracy': acc, 'f1': f1, 'preds': preds,
        'cm': confusion_matrix(targets, preds)
    }
    print(f'  {label:<32}  {acc*100:>7.2f}%  {f1:>10.4f}')


  Model                            Accuracy    Macro F1
----------------------------------------------------------


KeyError: 'efficientnet'

In [ ]:
# 12. Weighted Soft-Voting Ensemble
raw_w  = {k: max(histories[k]['val_acc']) for k in trained}
total  = sum(raw_w.values())
weights = {k: v / total for k, v in raw_w.items()}

print('Ensemble weights:')
for k, w in weights.items():
    print(f'  {LABELS[k]:<30} {w:.3f}  (best val {raw_w[k]:.2f}%)')

ensemble_probs = sum(weights[k] * all_probs[k] for k in trained)
ensemble_preds = ensemble_probs.argmax(axis=1)

ens_acc = accuracy_score(targets, ensemble_preds)
_, _, ens_f1, _ = precision_recall_fscore_support(
    targets, ensemble_preds, average='macro', zero_division=0)
ens_cm = confusion_matrix(targets, ensemble_preds)

best_individual = max(r['accuracy'] for r in ind_results.values())
gain = (ens_acc - best_individual) * 100

print(f'\nENSEMBLE TEST RESULTS')
print(f'  Accuracy  : {ens_acc*100:.2f}%')
print(f'  Macro F1  : {ens_f1:.4f}')
print(f'  Gain over best individual model: +{gain:.2f}%')
print('\n', classification_report(targets, ensemble_preds,
                                    target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# 13. Visualisations
COLORS = {'efficientnet': '#4C72B0', 'convnext': '#DD8452', 'vit': '#55A868'}
SHORT  = [c.split()[0] for c in CLASS_NAMES]

fig = plt.figure(figsize=(22, 12))

ax = fig.add_subplot(2, 4, 1)
for k, h in histories.items():
    ax.plot(range(1, len(h['val_acc'])+1), h['val_acc'],
            label=LABELS[k], color=COLORS[k], lw=2)
ax.set(title='Validation Accuracy', xlabel='Epoch', ylabel='Acc %')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = fig.add_subplot(2, 4, 2)
for k, h in histories.items():
    ax.plot(range(1, len(h['val_loss'])+1), h['val_loss'],
            label=LABELS[k], color=COLORS[k], lw=2)
ax.set(title='Validation Loss', xlabel='Epoch', ylabel='Loss')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = fig.add_subplot(2, 4, 3)
names   = list(LABELS.values()) + ['Ensemble']
accs    = [ind_results[k]['accuracy']*100 for k in LABELS] + [ens_acc*100]
bcolors = list(COLORS.values()) + ['#C44E52']
bars = ax.bar(names, accs, color=bcolors, alpha=0.85, edgecolor='white', width=0.55)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
            f'{a:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set(title='Test Accuracy', ylabel='Accuracy (%)')
ax.tick_params(axis='x', rotation=18); ax.grid(axis='y', alpha=0.3)

ax = fig.add_subplot(2, 4, 4)
f1s = [ind_results[k]['f1'] for k in LABELS] + [ens_f1]
bars = ax.bar(names, f1s, color=bcolors, alpha=0.85, edgecolor='white', width=0.55)
for b, f in zip(bars, f1s):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
            f'{f:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set(title='Macro F1 Score', ylabel='F1')
ax.tick_params(axis='x', rotation=18); ax.grid(axis='y', alpha=0.3)

cms       = {k: ind_results[k]['cm'] for k in LABELS}
cms['ens'] = ens_cm
cm_keys   = list(LABELS.keys()) + ['ens']
cm_titles = list(LABELS.values()) + ['Ensemble']
for idx, (key, title) in enumerate(zip(cm_keys, cm_titles)):
    ax = fig.add_subplot(2, 4, 5 + idx)
    cm_n = cms[key].astype(float) / (cms[key].sum(axis=1, keepdims=True) + 1e-9)
    sns.heatmap(cm_n, annot=False, cmap='Blues', ax=ax, vmin=0, vmax=1,
                xticklabels=SHORT, yticklabels=SHORT, cbar=False)
    ax.set_title(title, fontsize=9); ax.tick_params(labelsize=7)

plt.suptitle('Eye Disease - T4 Balanced Ensemble Results',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# 14. Save Models and Ensemble Config
for key in trained:
    torch.save(trained[key].state_dict(), f'{key}.pth')

with open('ensemble_config.json', 'w') as f:
    json.dump({'weights': weights, 'img_size': IMG_SIZE,
               'class_names': CLASS_NAMES}, f, indent=2)

print('Saved: efficientnet.pth | convnext.pth | vit.pth | ensemble_config.json')

print('\nFINAL SUMMARY')
print('=' * 60)
for k, label in LABELS.items():
    r = ind_results[k]
    print(f'  {label:<30} Acc={r["accuracy"]*100:.2f}%  F1={r["f1"]:.4f}')
print(f'  {"Ensemble":<30} Acc={ens_acc*100:.2f}%  F1={ens_f1:.4f}  (+{gain:.2f}%)')
